# Bus Passenger-Demand Forecasting — Exploratory Data Analysis (EDA)
**Algions AI Labs Private Limited | AI & Machine Learning Advanced Assessment**

This notebook conducts end-to-end exploratory data analysis for trip-level bus ridership forecasting, examining:
1. Data distributions and summary statistics
2. Data quality, missingness, and leakage safety
3. Hourly and day-of-week demand patterns (commuter peaks vs. leisure surges)
4. Route-level heterogeneity across 6 operational corridors
5. Advance booking conversion rates at D-1 cut-off

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
data_path = Path("../data/raw/bus_trips_historical.csv")
df = pd.read_csv(data_path)
df["departure_datetime"] = pd.to_datetime(df["departure_datetime"])
print(f"Total trips loaded: {len(df):,}")
print(f"Date span: {df['departure_datetime'].min()} to {df['departure_datetime'].max()}")

## 1. Dataset Profiling and Quality Inspection

In [2]:
df.info()
df.describe().round(2)

## 2. Route-Level Demand Heterogeneity
Compare passenger boardings across commuter, university, intercity, airport, tourist, and feeder routes.

In [3]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="route_id", y="passenger_count", hue="route_id", palette="Set2", legend=False)
plt.title("Passenger Demand by Route Identifier", fontsize=13, fontweight="bold")
plt.xlabel("Route ID")
plt.ylabel("Boarded Passengers")
plt.show()

## 3. Hourly Commuter Rush-Hour Profiles
Analysis of morning (07:00-09:30) and evening (16:30-19:30) peak travel vs off-peak.

In [4]:
hourly_agg = df.groupby(["departure_hour", "is_weekend"])["passenger_count"].mean().reset_index()
hourly_agg["Day Type"] = hourly_agg["is_weekend"].map({0: "Weekday (Mon-Fri)", 1: "Weekend (Sat-Sun)"})

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_agg, x="departure_hour", y="passenger_count", hue="Day Type", marker="o", lw=2.5)
plt.title("Hourly Demand Profile: Weekdays vs. Weekends", fontsize=13, fontweight="bold")
plt.xlabel("Hour of Departure (24h)")
plt.ylabel("Average Passenger Count")
plt.xticks(range(5, 23))
plt.show()

## 4. D-1 Advance Booking Signal Strength
Evaluating the linear and non-linear correlation between bookings at D-1 18:00 cut-off and final passengers.

In [5]:
corr = df["advance_bookings_d1"].corr(df["passenger_count"])
print(f"Pearson correlation: {corr:.4f}")

plt.figure(figsize=(8, 5))
plt.scatter(df["advance_bookings_d1"], df["passenger_count"], alpha=0.2, color="#2980b9")
plt.title(f"D-1 Advance Bookings vs Final Boarded Passengers (r = {corr:.3f})", fontsize=13, fontweight="bold")
plt.xlabel("Confirmed Bookings (D-1 18:00)")
plt.ylabel("Actual Passengers Boarded")
plt.show()